# 查询参数模型

如果你有一组具有相关性的查询参数，你可以创建一个 Pydantic 模型来声明它们。

这将允许你在多个地方去复用模型，并且一次性为所有参数声明验证和元数据。😎

> FastAPI 从 0.115.0 版本开始支持这个特性。🤓

## 使用 Pydantic 模型的查询参数
在一个 Pydantic 模型中声明你需要的查询参数，然后将参数声明为 `Query`：

In [ ]:
from typing import Annotated, Literal

from fastapi import FastAPI, Query
from pydantic import BaseModel, Field

app = FastAPI()


class FilterParams(BaseModel):
    limit: int = Field(100, gt=0, le=100)
    offset: int = Field(0, ge=0)
    order_by: Literal["created_at", "updated_at"] = "created_at"
    tags: list[str] = []


@app.get("/items/")
async def read_items(filter_query: Annotated[FilterParams, Query()]):
    return filter_query

FastAPI 将会从请求的查询参数中提取出每个字段的数据，并将其提供给你定义的 Pydantic 模型。

## 查看文档

你可以在 `/docs` 页面的 `UI` 中查看查询参数：

<img src="https://fastapi.tiangolo.com/img/tutorial/query-param-models/image01.png">

## 禁止额外的查询参数¶
在一些特殊的使用场景中（可能不是很常见），你可能希望限制你要接收的查询参数。

你可以使用 Pydantic 的模型配置来 forbid（意为禁止 —— 译者注）任何 extra（意为额外的 —— 译者注）字段：

In [ ]:
from typing import Annotated, Literal

from fastapi import FastAPI, Query
from pydantic import BaseModel, Field

app = FastAPI()


class FilterParams(BaseModel):
    model_config = {"extra": "forbid"}

    limit: int = Field(100, gt=0, le=100)
    offset: int = Field(0, ge=0)
    order_by: Literal["created_at", "updated_at"] = "created_at"
    tags: list[str] = []


@app.get("/items/")
async def read_items(filter_query: Annotated[FilterParams, Query()]):
    return filter_query

假设有一个客户端尝试在查询参数中发送一些额外的数据，它将会收到一个错误响应。

例如，如果客户端尝试发送一个值为 `plumbus` 的 `tool` 查询参数，如：

`https://example.com/items/?limit=10&tool=plumbus`

他们将收到一个错误响应，告诉他们查询参数 `tool` 是不允许的：

```json
{
    "detail": [
        {
            "type": "extra_forbidden",
            "loc": ["query", "tool"],
            "msg": "Extra inputs are not permitted",
            "input": "plumbus"
        }
    ]
}
```

## 总结
你可以使用 Pydantic 模型在 FastAPI 中声明查询参数。